In [2]:
import pandas as pd
from pathlib import Path
print(pd.__version__)

3.0.3


In [3]:
eia_path = Path("../data/raw/eia860/2024/2___Plant_Y2024.xlsx")
eia_path.exists()

True

In [4]:
import os
os.getcwd()

'c:\\Projects\\pjm-data-center-siting\\notebooks'

In [5]:
plants = pd.read_excel(eia_path, skiprows=1)
plants.shape

(16132, 42)

In [6]:
import sys
print(sys.executable)

c:\Projects\pjm-data-center-siting\.venv\Scripts\python.exe


In [8]:
plants.columns.tolist()

['Utility ID',
 'Utility Name',
 'Plant Code',
 'Plant Name',
 'Street Address',
 'City',
 'State',
 'Zip',
 'County',
 'Latitude',
 'Longitude',
 'NERC Region',
 'Balancing Authority Code',
 'Balancing Authority Name',
 'Name of Water Source',
 'Primary Purpose (NAICS Code)',
 'Regulatory Status',
 'Sector',
 'Sector Name',
 'FERC Cogeneration Status',
 'FERC Cogeneration Docket Number',
 'FERC Small Power Producer Status',
 'FERC Small Power Producer Docket Number',
 'FERC Exempt Wholesale Generator Status',
 'FERC Exempt Wholesale Generator Docket Number',
 'Ash Impoundment?',
 'Ash Impoundment Lined?',
 'Ash Impoundment Status',
 'Transmission or Distribution System Owner',
 'Transmission or Distribution System Owner ID',
 'Transmission or Distribution System Owner State',
 'Grid Voltage (kV)',
 'Grid Voltage 2 (kV)',
 'Grid Voltage 3 (kV)',
 'Energy Storage',
 'Natural Gas LDC Name',
 'Natural Gas Pipeline Name 1',
 'Natural Gas Pipeline Name 2',
 'Natural Gas Pipeline Name 3',
 '

In [9]:
pjm_plants = plants[plants["Balancing Authority Code"] == "PJM"]
pjm_plants.shape

(2234, 42)

In [10]:
pjm_plants[["Plant Name", "State", "County", "Utility Name", "Grid Voltage (kV)"]].head(20)

,Plant Name,State,County,Utility Name,Grid Voltage (kV)
30,J K Smith,KY,Clark,"East Kentucky Power Coop, Inc",138
235,Joliet 29,IL,Will,Midwest Generations EME LLC,345
380,Christiana,DE,New Castle,Calpine Mid-Atlantic Generation LLC,12.5
381,Delaware City 10,DE,New Castle,Calpine Mid-Atlantic Generation LLC,13.8
382,Edge Moor,DE,New Castle,Calpine Mid-Atlantic Generation LLC,230
383,Indian River Generating Station,DE,Sussex,Indian River Operations Inc,69
384,West Station (DE),DE,New Castle,Calpine Mid-Atlantic Generation LLC,12
385,McKee Run,DE,Kent,NAES Corporation - (DE),230
386,Brandon Shores,MD,Anne Arundel,Brandon Shores LLC,230
536,Crawford,IL,Cook,Midwest Generations EME LLC,138


In [11]:
pjm_plants["Grid Voltage (kV)"].value_counts().head(20)

Grid Voltage (kV)
34.5     331
12.47    266
138      211
69       176
13.2     146
230      113
345      108
0.48      96
115       93
12.5      65
12        62
13.8      57
500       40
13        37
26.4      34
34        30
480       26
23        25
15        22
25        21
Name: count, dtype: int64

In [12]:
pjm_plants["State"].value_counts()

State
NJ    423
PA    322
VA    314
IL    281
OH    261
MD    246
NC    131
IN     76
WV     56
DE     43
KY     42
DC     22
MI     15
TN      1
MN      1
Name: count, dtype: int64

In [13]:
pjm_plants["Transmission or Distribution System Owner"].value_counts().head(20)

Transmission or Distribution System Owner
Virginia Electric & Power Co         316
Commonwealth Edison Co               259
Public Service Elec & Gas Co         215
Jersey Central Power & Lt Co         139
Baltimore Gas & Electric Co          102
Potomac Electric Power Co             75
PPL Electric Utilities Corp           74
Delmarva Power                        73
Ohio Power Co                         64
Pennsylvania Electric Co              63
Indiana Michigan Power Co             61
PECO Energy Co                        54
Atlantic City Electric Co             53
Appalachian Power Co                  46
Metropolitan Edison Co                31
West Penn Power Company               31
The Potomac Edison Company            29
American Transmission Systems Inc     28
Duquesne Light Co                     24
Monongahela Power Co                  23
Name: count, dtype: int64

In [14]:
transmission_voltages = [138, 230, 345, 500, 765]
pjm_transmission = pjm_plants[pjm_plants["Grid Voltage (kV)"].isin(transmission_voltages)]
pjm_transmission.shape

(480, 42)

In [15]:
pjm_transmission.groupby(["State", "Grid Voltage (kV)"]).size().unstack(fill_value=0)

Grid Voltage (kV),138,230,345,500,765
State,,,,,
DE,4,4,0,0,0
IL,37,0,37,0,0
IN,14,0,21,0,1
KY,8,0,5,0,0
MD,13,10,0,3,0
MI,0,0,3,0,0
NC,0,9,0,0,0
NJ,6,20,1,3,0
OH,50,0,37,0,4


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("PJM_API_KEY")
print(f"Key loaded: {api_key[:8]}..." if api_key else "Key not loaded")

Key loaded: 8375fc51...


In [2]:
import requests

base_url = "https://api.pjm.com/api/v1"
headers = {"Ocp-Apim-Subscription-Key": api_key}

response = requests.get(f"{base_url}/da_hrl_lmps/metadata", headers=headers)
print(f"Status code: {response.status_code}")
print(f"Response keys: {list(response.json().keys()) if response.status_code == 200 else response.text[:500]}")

Status code: 200
Response keys: ['id', 'feedId', 'name', 'displayName', 'description', 'category', 'firstAvailable', 'version', 'internalName', 'defaultSearchJson', 'fileLoadingMethod', 'postingFrequency', 'postingDay', 'retentionTime', 'lastDataLoad', 'lastDataLoadEst', 'isFrequentlyAccessed', 'enableArchiving', 'archiveCutoffDays', 'created', 'updated', 'repostNotifications', 'columns', 'links', 'timeDurationUnit', 'timeDurationIncrement', 'categoryOrdinal', 'feedProductType']


In [3]:
metadata = response.json()
print(f"First available: {metadata['firstAvailable']}")
print(f"Archive enabled: {metadata['enableArchiving']}")
print(f"Archive cutoff days: {metadata['archiveCutoffDays']}")
print(f"Last data load (EST): {metadata['lastDataLoadEst']}")
print(f"Posting frequency: {metadata['postingFrequency']}")
print(f"Number of columns: {len(metadata['columns'])}")

First available: 2000-06-01T00:00:00
Archive enabled: True
Archive cutoff days: 731
Last data load (EST): 2026-05-27T13:09:22.5866667-04:00
Posting frequency: Daily
Number of columns: 14


In [4]:
params = {
    "rowCount": 24,
    "startRow": 1,
    "datetime_beginning_ept": "2024-01-15 00:00 to 2024-01-15 23:59",
    "pnode_id": 51288,
    "row_is_current": "true",
    "format": "json"
}

response = requests.get(f"{base_url}/da_hrl_lmps", headers=headers, params=params)
print(f"Status code: {response.status_code}")
print(f"Total rows available: {response.headers.get('X-TotalRows', 'not in header')}")

if response.status_code == 200:
    data = response.json()
    print(f"Top-level keys: {list(data.keys())}")
    if "items" in data:
        print(f"Number of items returned: {len(data['items'])}")
        print(f"First item: {data['items'][0]}")

Status code: 400
Total rows available: not in header


In [5]:
print(f"Status code: {response.status_code}")
print(f"Response body: {response.text}")

Status code: 400
Response body: {"errors":[{"field":"Filters","message":"The API request contains invalid attribute(s) for archived data - Pnode_Id. Please update the request and retry.","detail":["pnode_id"]}],"feedMetadata":{"id":45,"feedId":"6","name":"da_hrl_lmps","displayName":"Day-Ahead Hourly LMPs","description":"This feed contains hourly Day-Ahead Energy Market locational marginal pricing (LMP) data for all bus locations, including aggregates. RSS Notification for this feed contains Market Day. To enhance the performance of current data queries and to better handle the increasing volume, PJM has implemented an archiving solution for this posting. The archived data is still available, but has slightly less flexibility on querying parameters than the more recent data does. The details of the restrictions and expected results for several filter combinations are detailed in the https://pjm.com/markets-and-operations/etools/data-miner-2.aspx. Data for this feed is archived after two

In [6]:
params = {
    "rowCount": 24,
    "startRow": 1,
    "datetime_beginning_ept": "2026-05-26 00:00 to 2026-05-26 23:59",
    "pnode_id": 51288,
    "row_is_current": "true",
    "format": "json"
}

response = requests.get(f"{base_url}/da_hrl_lmps", headers=headers, params=params)
print(f"Status code: {response.status_code}")
print(f"Total rows available: {response.headers.get('X-TotalRows', 'not in header')}")
print(f"Response body (first 500 chars): {response.text[:500]}")

Status code: 200
Total rows available: not in header
Response body (first 500 chars): {"links":[{"rel":"self","href":"https://api.pjm.com/api/v1/da_hrl_lmps?RowCount=24&Order=Asc&StartRow=1&IsActiveMetadata=True&Fields=congestion_price_da%2Cdatetime_beginning_ept%2Cdatetime_beginning_utc%2Cequipment%2Cmarginal_loss_price_da%2Cpnode_id%2Cpnode_name%2Crow_is_current%2Csystem_energy_price_da%2Ctotal_lmp_da%2Ctype%2Cversion_nbr%2Cvoltage%2Czone&datetime_beginning_ept=2026-05-26%2000%3A00%20to%202026-05-26%2023%3A59&pnode_id=51288&row_is_current=true"},{"rel":"metadata","href":"https:


In [7]:
data = response.json()
print(f"Top-level keys: {list(data.keys())}")
print(f"Number of items: {len(data['items'])}")
print(f"\nFirst row:")
for key, value in data['items'][0].items():
    print(f"  {key}: {value}")

Top-level keys: ['links', 'items', 'searchSpecification', 'totalRows']
Number of items: 24

First row:
  datetime_beginning_utc: 2026-05-26T04:00:00
  datetime_beginning_ept: 2026-05-26T00:00:00
  pnode_id: 51288
  pnode_name: WESTERN HUB
  voltage: None
  equipment: None
  type: HUB
  zone: None
  system_energy_price_da: 25.36
  total_lmp_da: 26.506355
  congestion_price_da: 0.841248
  marginal_loss_price_da: 0.305107
  row_is_current: True
  version_nbr: 1
